In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# =============================================================================
# =============================================================================
mat_path = get_dataset_path("CETL_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat  = data["processed_Qd_mat"].astype(float)
processed_Vd_mat  = data["processed_Vd_mat"].astype(float)

n_rows, n_cells = processed_Qd_mat.shape
Qmax_all = processed_Qd_mat[-1, :].ravel()
n_samples = processed_Qd_mat.shape[1]

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
nature_colors = [
    "#313695",
    "#74add1",
    "#e0f3f8",
    "#fee090",
    "#f46d43",
    "#a50026",
]
cmap_nature = LinearSegmentedColormap.from_list("Nature_Style", nature_colors, N=256)
norm_nature = Normalize(vmin=np.nanmin(Qmax_all), vmax=np.nanmax(Qmax_all))

# =============================================================================
# =============================================================================
Vtail_low = 2.50
V_tail_upp = 2.73

print("Generating the tail-fragment figure...")
fig_tail, ax_tail = plt.subplots(figsize=(4, 6), constrained_layout=True)

for i in range(n_samples):
    Vd = processed_Vd_mat[:, i]
    Qd = processed_Qd_mat[:, i]
    mask = (Vd >= Vtail_low) & (Vd <= V_tail_upp)
    if not np.any(mask): continue
    
    ax_tail.plot(Qd[mask], Vd[mask], color=cmap_nature(norm_nature(Qmax_all[i])), linewidth=1.3, alpha=0.9)

ax_tail.axis('off')

fig_tail.savefig("fig1_tail_curve_2.50_2.73.png", dpi=300, transparent=True, bbox_inches="tight")
plt.close(fig_tail)


# =============================================================================
# =============================================================================
Vseg_low = V_tail_upp
Vseg_upp = V_tail_upp + 0.30
nun_seg = 6
V_edges = np.linspace(Vseg_low, Vseg_upp, nun_seg + 1)

print("Generating the six upstream-fragment figures...")
for seg_idx in range(nun_seg):
    fig_up, ax_up = plt.subplots(figsize=(6, 2), constrained_layout=True)
    
    V_low_k = V_edges[seg_idx]
    V_upp_k = V_edges[seg_idx + 1]

    for i in range(n_samples):
        Vd = processed_Vd_mat[:, i]
        Qd = processed_Qd_mat[:, i]
        mask = (Vd >= V_low_k) & (Vd <= V_upp_k)
        if not np.any(mask): continue
        
        ax_up.plot(Qd[mask], Vd[mask], color=cmap_nature(norm_nature(Qmax_all[i])), linewidth=1.0, alpha=0.9)

    ax_up.axis('off')

    fig_up.savefig(f"fig2_upstream_segment_{seg_idx+1}.png", dpi=300, transparent=True, bbox_inches="tight")
    plt.close(fig_up)


# =============================================================================
# =============================================================================
print("Generating the standalone colorbar figure...")
fig_cbar, ax_cbar = plt.subplots(figsize=(5, 1))

sm = ScalarMappable(norm=norm_nature, cmap=cmap_nature)
sm.set_array([])

cbar = fig_cbar.colorbar(sm, cax=ax_cbar, orientation="horizontal")

cbar.set_ticks([])
cbar.outline.set_visible(False)

fig_cbar.savefig("fig3_standalone_colorbar.png", dpi=300, transparent=True, bbox_inches="tight")
plt.close(fig_cbar)

print("\nAll eight figures have been generated and saved.")
